# Projeto de Sala, Dashboard de Vendas de Jogos

## 1. Seleção de dados e justificativa

Foi utilizada a base **Video Games Sales (1980-2024) - Raw.csv**, pois ela possui maior volume de registros e permite demonstrar de forma clara as etapas de **análise exploratória, tratamento de dados e construção de visualizações**.  
A escolha da base bruta é adequada ao objetivo do projeto porque possibilita identificar problemas reais de qualidade dos dados e justificar tecnicamente cada decisão de limpeza.

## 2. Importação das bibliotecas e carregamento da base

Como o projeto precisa obter os dados diretamente do **Kaggle**, a leitura da base será feita com `kagglehub`. O arquivo utilizado é **Video Games Sales (1980-2024) - Raw.csv** dentro do dataset `bhushandivekar/video-game-sales-and-industry-data-1980-2024`.

No VSCode ou em outros ambientes fora do Kaggle, pode ser necessário instalar a biblioteca antes de executar a próxima célula.


In [ ]:
# Execute esta célula apenas se o ambiente ainda não tiver o kagglehub instalado
%pip install "kagglehub[pandas-datasets]"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

sb.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

file_path = "Video Games Sales (1980-2024) - Raw.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "bhushandivekar/video-game-sales-and-industry-data-1980-2024",
    file_path
)

df.head()


## 3. Análise exploratória inicial

Aqui são verificadas a dimensão da base, os tipos das colunas, estatísticas descritivas, quantidade de duplicados e volume de valores ausentes.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False).round(2)

## 4. Tratamento de dados

As decisões adotadas foram:

- remoção de registros duplicados;
- conversão das colunas de data;
- criação da coluna `release_year`;
- preenchimento dos valores nulos da coluna `developer` com **"Unknown"**;
- preenchimento de `critic_score` com a mediana do gênero e, quando necessário, com a mediana global;
- preenchimento das vendas regionais com a mediana do gênero e, quando necessário, com a mediana global;
- preenchimento de `total_sales` pela soma das vendas regionais quando ausente;
- preenchimento de `release_date` faltante com uma data neutra do ano imputado;
- preenchimento de `last_update` com `release_date` quando ausente.

Assim, a base final fica sem valores nulos, com um tratamento simples e fácil de explicar na apresentação.


In [ ]:
df = df.copy()

df = df.drop_duplicates()

df["developer"] = df["developer"].fillna("Unknown")

df["release_date"] = pd.to_datetime(df["release_date"], dayfirst=True, errors="coerce")
df["last_update"] = pd.to_datetime(df["last_update"], dayfirst=True, errors="coerce")
df["release_year"] = df["release_date"].dt.year

genre_year_median = df.groupby("genre")["release_year"].median().round()
global_year = int(round(df["release_year"].median()))

df["release_year"] = df["release_year"].fillna(df["genre"].map(genre_year_median))
df["release_year"] = df["release_year"].fillna(global_year).astype(int)

missing_release_date = df["release_date"].isna()
df.loc[missing_release_date, "release_date"] = pd.to_datetime(
    df.loc[missing_release_date, "release_year"].astype(str) + "-07-01"
)

df["last_update"] = df["last_update"].fillna(df["release_date"])

genre_score_median = df.groupby("genre")["critic_score"].median()
global_score = round(df["critic_score"].median(), 2)

df["critic_score"] = df["critic_score"].fillna(df["genre"].map(genre_score_median))
df["critic_score"] = df["critic_score"].fillna(global_score).round(2)

df["total_sales"] = pd.to_numeric(df["total_sales"], errors="coerce")
df["na_sales"] = pd.to_numeric(df["na_sales"], errors="coerce")
df["jp_sales"] = pd.to_numeric(df["jp_sales"], errors="coerce")
df["pal_sales"] = pd.to_numeric(df["pal_sales"], errors="coerce")
df["other_sales"] = pd.to_numeric(df["other_sales"], errors="coerce")

genre_na_median = df.groupby("genre")["na_sales"].median()
genre_jp_median = df.groupby("genre")["jp_sales"].median()
genre_pal_median = df.groupby("genre")["pal_sales"].median()
genre_other_median = df.groupby("genre")["other_sales"].median()

global_na = df["na_sales"].median()
global_jp = df["jp_sales"].median()
global_pal = df["pal_sales"].median()
global_other = df["other_sales"].median()

df["na_sales"] = df["na_sales"].fillna(df["genre"].map(genre_na_median))
df["jp_sales"] = df["jp_sales"].fillna(df["genre"].map(genre_jp_median))
df["pal_sales"] = df["pal_sales"].fillna(df["genre"].map(genre_pal_median))
df["other_sales"] = df["other_sales"].fillna(df["genre"].map(genre_other_median))

df["na_sales"] = df["na_sales"].fillna(global_na)
df["jp_sales"] = df["jp_sales"].fillna(global_jp)
df["pal_sales"] = df["pal_sales"].fillna(global_pal)
df["other_sales"] = df["other_sales"].fillna(global_other)

df["total_sales"] = df["total_sales"].fillna(
    df["na_sales"] + df["jp_sales"] + df["pal_sales"] + df["other_sales"]
).round(2)

global_total = df["total_sales"].median()
df["total_sales"] = df["total_sales"].fillna(global_total).round(2)

df["release_date"] = df["release_date"].dt.strftime("%Y-%m-%d")
df["last_update"] = df["last_update"].dt.strftime("%Y-%m-%d")

df.isna().sum().sort_values(ascending=False)

In [ ]:
print("Total de nulos restantes:", int(df.isna().sum().sum()))
print("Dimensão final da base:", df.shape)
print("Colunas finais:", df.columns.tolist())

## 5. Tabela de qualidade dos dados após o tratamento

In [ ]:
qualidade = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "percentual_nulos": (df.isnull().mean() * 100).round(2),
    "tipo": df.dtypes.astype(str)
}).sort_values("percentual_nulos", ascending=False)

qualidade

## 6. Criação de subconjuntos para análise

Como a base tratada ficou sem valores nulos, os subconjuntos abaixo passam a ser recortes temáticos para organizar as análises:

- `df_vendas`: análises gerais de vendas;
- `df_critica`: relação entre nota da crítica e vendas;
- `df_tempo`: análise temporal por ano.


In [ ]:
df_vendas = df.copy()
df_critica = df.copy()
df_tempo = df.copy()

print("Base completa:", df.shape)
print("Base de vendas:", df_vendas.shape)
print("Base de crítica:", df_critica.shape)
print("Base temporal:", df_tempo.shape)

## 7. Análises exploratórias principais

In [ ]:
df_vendas.groupby("genre")["total_sales"].sum().sort_values(ascending=False).head(10)

In [ ]:
df_vendas.groupby("console")["total_sales"].sum().sort_values(ascending=False).head(10)

In [ ]:
df_vendas.groupby("publisher")["total_sales"].sum().sort_values(ascending=False).head(10)

In [ ]:
df_tempo.groupby("release_year")["total_sales"].sum().sort_values(ascending=False).head(10)

In [ ]:
df_vendas[["title", "console", "genre", "publisher", "developer", "total_sales"]] \
    .sort_values("total_sales", ascending=False).head(10)

## 8. Visualizações para o dashboard

In [ ]:
vendas_genero = df_vendas.groupby("genre")["total_sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sb.barplot(x=vendas_genero.index, y=vendas_genero.values)
plt.xticks(rotation=45)
plt.title("Vendas totais por gênero")
plt.xlabel("Gênero")
plt.ylabel("Total de vendas")
plt.show()

In [ ]:
vendas_console = df_vendas.groupby("console")["total_sales"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sb.barplot(x=vendas_console.index, y=vendas_console.values)
plt.xticks(rotation=45)
plt.title("Top 10 consoles por vendas")
plt.xlabel("Console")
plt.ylabel("Total de vendas")
plt.show()

In [ ]:
vendas_publisher = df_vendas.groupby("publisher")["total_sales"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sb.barplot(x=vendas_publisher.index, y=vendas_publisher.values)
plt.xticks(rotation=45)
plt.title("Top 10 publishers por vendas")
plt.xlabel("Publisher")
plt.ylabel("Total de vendas")
plt.show()

In [ ]:
vendas_ano = df_tempo.groupby("release_year")["total_sales"].sum().sort_index()

plt.figure(figsize=(12, 5))
plt.plot(vendas_ano.index, vendas_ano.values)
plt.title("Vendas totais por ano")
plt.xlabel("Ano")
plt.ylabel("Total de vendas")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sb.scatterplot(data=df_critica, x="critic_score", y="total_sales")
plt.title("Relação entre nota da crítica e vendas totais")
plt.xlabel("Nota da crítica")
plt.ylabel("Total de vendas")
plt.show()

In [ ]:
df_critica[["critic_score", "total_sales"]].corr()

## 9. Conclusões parciais para a apresentação

Com base nas análises, a dupla poderá destacar:

- quais consoles concentraram mais vendas;
- quais gêneros tiveram maior desempenho comercial;
- quais publishers lideraram o mercado;
- em quais anos houve pico de vendas;
- se existe ou não relação aparente entre nota da crítica e vendas totais.

## 10. Exportação da base tratada

Esta etapa gera um arquivo pronto para ser usado no dashboard final.

In [ ]:
df.to_csv("video_games_sales_tratado.csv", index=False)